In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/01000
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  1612.5488218670068
RUN  2 , total integrated cost =  1523.5826140203562
RUN  3 , total integrated cost =  1403.0458118738318
RUN  4 , total integrated cost =  1318.5172528888756
RUN  5 , total integrated cost =  1189.034969337594
RUN  6 , total integrated cost =  1090.8798135491857
RUN  7 , total integrated cost =  874.0642873258037
RUN  8 , total integrated cost =  739.4555526246942
RUN  9 , total integrated cost =  532.1297626250068
RUN  10 , total integrated cost =  440.6745306122127
RUN  11 , total integrated cost =  260.4168029855361
RUN  12 , total integrated cost =  207.47718339780783
RUN  13 , total integrated cost =  169.91827378819625
RUN  14 , total integrated cost =  150.24376211774415
RUN  15 , total integrated cost =  139.94288143054771
RUN

ERROR:root:Problem in initial value trasfer


RUN  300 , total integrated cost =  55.32135889188139
Control only changes marginally.
RUN  305 , total integrated cost =  55.32135889188063
Improved over  305  iterations in  17.436123326420784  seconds by  98.91469073259624  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446695382865 -56.62446683751675
weight =  921.3963522049056
set cost params:  1.0 921.3963522049056 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5087.733977614825
Gradient descend method:  None
RUN  1 , total integrated cost =  4986.017555821411
RUN  2 , total integrated cost =  4982.445898399989
RUN  3 , total integrated cost =  4952.425443783786
RUN  4 , total integrated cost =  4940.951221919552
RUN  5 , total integrated cost =  4940.070006572746
RUN  6 , total integrated cost =  4885.616731819233
RUN  7 , total integrated cost =  4884.953600370108
RUN  8 , total integrated cost =  4884.938227012016
RUN  9 , total integrated cost =  4884.936604209078
RUN  10 , total inte

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  25 , total integrated cost =  4884.936242112655
Improved over  25  iterations in  0.5820954367518425  seconds by  3.9860129557568342  percent.
Problem in initial value trasfer:  Vmean_exc -56.624826590097314 -56.62480979597332
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  3453.0228318477593
RUN  2 , total integrated cost =  2139.549951980828
RUN  3 , total integrated cost =  1573.4043212729177
RUN  4 , total integrated cost =  1114.2789355940379
RUN  5 , total integrated cost =  898.0726154949658
RUN  6 , total integrated cost =  692.7906432930969
RUN  7 , total integrated cost =  573.5563917323714
RUN  8 , total integrated cost =  459.1313183339911
RUN  9 , total integrated cost =  395.25408434421274
RUN  10 , total integrated cost =  311.80756946166537
RUN  1

...

RUN  1 , total integrated cost =  5845.286879790712
Control only changes marginally.
RUN  1 , total integrated cost =  5845.286879790712
Improved over  1  iterations in  0.04413002170622349  seconds by  0.0  percent.
-------  125 0.47500000000000014 0.8500000000000005
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  14547.979043359295
Gradient descend method:  None
RUN  1 , total integrated cost =  14547.979043359166
RUN  2 , total integrated cost =  14547.979043359146
RUN  3 , total integrated cost =  14547.979043359146
Control only changes marginally.
RUN  3 , total integrated cost =  14547.979043359146
Improved over  3  iterations in  0.09636232256889343  seconds by  1.0231815394945443e-12  percent.
Problem in initial value trasfer:  Vmean_exc -78.45982004642322 -78.45982007482613
weight =  10.000000000000103
set cost params:  1.0 10.000000000000103 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  14547

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  23532.63613173007
RUN  2 , total integrated cost =  23532.636131730058
RUN  3 , total integrated cost =  23532.636131730058
Control only changes marginally.
RUN  3 , total integrated cost =  23532.636131730058
Improved over  3  iterations in  0.11056683026254177  seconds by  4.8290061727129796e-08  percent.
Problem in initial value trasfer:  Vmean_exc -73.65450099847658 -73.65451701680384
weight =  10.000000004829007
set cost params:  1.0 10.000000004829007 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23532.636131730058
Gradient descend method:  None
RUN  1 , total integrated cost =  23532.636131730058
Control only changes marginally.
RUN  1 , total integrated cost =  23532.636131730058
Improved over  1  iterations in  0.05959131568670273  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -73.65450099847658 -73.65451701680384
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.

ERROR:root:Problem in initial value trasfer


RUN  40 , total integrated cost =  32315.852383992238
Control only changes marginally.
RUN  43 , total integrated cost =  32315.852383992184
Improved over  43  iterations in  0.9966884329915047  seconds by  2.875753071650479  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354415390001 -56.70354406004581


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  8231.907221040567
Control only changes marginally.
RUN  7 , total integrated cost =  8231.907221040567
Improved over  7  iterations in  0.2275360580533743  seconds by  0.9158263079744273  percent.
Problem in initial value trasfer:  Vmean_exc -75.18587793278516 -75.18588098755109
weight =  10.000000000519405
set cost params:  1.0 10.000000000519405 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8231.907221040567
Gradient descend method:  None
RUN  1 , total integrated cost =  8231.907221040567
Control only changes marginally.
RUN  1 , total integrated cost =  8231.907221040567
Improved over  1  iterations in  0.057602835819125175  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -75.18587793278516 -75.18588098755109
-------  30 0.4250000000000001 0.5250000000000002
[0, 5, 10, 15, 20] []
closest index  20
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , 

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  7978.317181774852
Control only changes marginally.
RUN  6 , total integrated cost =  7978.317181774852
Improved over  6  iterations in  0.19170418195426464  seconds by  0.9526557954649917  percent.
Problem in initial value trasfer:  Vmean_exc -76.60098038480734 -76.60098083834617
weight =  10.000000000013573
set cost params:  1.0 10.000000000013573 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7978.317181774852
Gradient descend method:  None
RUN  1 , total integrated cost =  7978.317181774852
Control only changes marginally.
RUN  1 , total integrated cost =  7978.317181774852
Improved over  1  iterations in  0.05190429650247097  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -76.60098038480734 -76.60098083834617
-------  35 0.5500000000000003 0.5250000000000002
found solution for  35
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  45 0.5000000000000002 0.57500000000

...

RUN  3 , total integrated cost =  14457.019642475083
RUN  4 , total integrated cost =  14457.019625929453
RUN  5 , total integrated cost =  14457.019625527053
RUN  6 , total integrated cost =  14457.019625512257
RUN  7 , total integrated cost =  14457.019625511597
RUN  8 , total integrated cost =  14457.01962551159
RUN  9 , total integrated cost =  14457.019625511579
RUN  10 , total integrated cost =  14457.019625511579
Control only changes marginally.
RUN  10 , total integrated cost =  14457.019625511579
Improved over  10  iterations in  0.2639531046152115  seconds by  0.563015976035885  percent.
Problem in initial value trasfer:  Vmean_exc -56.6771004314507 -56.677104411771445
-------  130 0.6000000000000003 0.8500000000000005
-------  135 0.5250000000000001 0.8750000000000006
-------  140 0.4500000000000001 0.9000000000000006
[0, 5, 10, 15, 20, 35, 40, 45, 60, 75, 80, 90, 105, 120, 130, 145, 95, 65, 50, 55, 135, 70, 85, 100] [120, 145, 130, 95, 105, 135]
closest index  100
set cost 

ERROR:root:Problem in initial value trasfer


RUN  19 , total integrated cost =  308.2795459798828
RUN  20 , total integrated cost =  308.27954597988185
Control only changes marginally.
RUN  23 , total integrated cost =  308.2795459798814
Improved over  23  iterations in  0.5300886537879705  seconds by  97.00228911035428  percent.
Problem in initial value trasfer:  Vmean_exc -56.65160412410592 -56.65160496499076
weight =  325.02865173014703
set cost params:  1.0 325.02865173014703 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10010.974772932494
Gradient descend method:  None
RUN  1 , total integrated cost =  9980.496650817726
RUN  2 , total integrated cost =  9980.368950205993
RUN  3 , total integrated cost =  9980.36460625764
RUN  4 , total integrated cost =  9980.364292238655
RUN  5 , total integrated cost =  9980.364264196654
RUN  6 , total integrated cost =  9980.364261881452
RUN  7 , total integrated cost =  9980.364261675186
RUN  8 , total integrated cost =  9980.364261658608
RUN  9 , total inte

ERROR:root:Problem in initial value trasfer


RUN  13 , total integrated cost =  9980.364261657398
Control only changes marginally.
RUN  13 , total integrated cost =  9980.364261657398
Improved over  13  iterations in  0.33671630918979645  seconds by  0.3057695376264462  percent.
Problem in initial value trasfer:  Vmean_exc -56.65092947673226 -56.65094156329171
-------  145 0.5750000000000002 0.9000000000000006
------------------------------------------------------------
-------------------- 7
------------------------------------------------------------
found solution:  [0, 5, 10, 15, 20, 35, 40, 45, 60, 75, 80, 90, 105, 120, 130, 145, 95, 65, 50, 55, 135, 70, 85, 100]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
found solution for  25
-------  30 0.42500000000000

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  960.4504716244119
set cost params:  1.0 960.4504716244119 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5086.805724273477
Gradient descend method:  None
RUN  1 , total integrated cost =  5086.77266087298
RUN  2 , total integrated cost =  5086.76983248521
RUN  3 , total integrated cost =  5086.769436627023
RUN  4 , total integrated cost =  5086.76

ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  5086.769358372073
RUN  9 , total integrated cost =  5086.76935836775
RUN  10 , total integrated cost =  5086.769358367046
RUN  11 , total integrated cost =  5086.76935836693
RUN  12 , total integrated cost =  5086.769358366912
RUN  13 , total integrated cost =  5086.769358366906
RUN  14 , total integrated cost =  5086.7693583669
RUN  15 , total integrated cost =  5086.7693583669
Control only changes marginally.
RUN  15 , total integrated cost =  5086.7693583669
Improved over  15  iterations in  0.3814262803643942  seconds by  0.00071490653562023  percent.
Problem in initial value trasfer:  Vmean_exc -56.62481295451783 -56.62479635122376
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  2646.7638091751264
set cost params:  1.0 2646.7638091751264 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13004.91650756916
Gradient descend method

ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  13004.75268098894
RUN  9 , total integrated cost =  13004.75268098894
Control only changes marginally.
RUN  9 , total integrated cost =  13004.75268098894
Improved over  9  iterations in  0.2785395849496126  seconds by  0.0012597280430384217  percent.
Problem in initial value trasfer:  Vmean_exc -56.670672389549395 -56.67066913549048
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  527.9643667350679
set cost params:  1.0 527.9643667350679 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8216.164544667256
Gradient descend method:  None
RUN  1 , total integrated cost =  8216.164508102222
RUN  2 , total integrated cost =  8216.16446904412
RUN  3 , total integrated cost =  8216.164418466891
RUN  4 , total integrated cost =  8216.164365436809
RUN  5 , total integrated cost =  8216.164333665096
RUN  6 , total integrated cost =  8216.164314

...

RUN  2 , total integrated cost =  24119.308932789114
RUN  3 , total integrated cost =  24119.308932789114
Control only changes marginally.
RUN  3 , total integrated cost =  24119.308932789114
Improved over  3  iterations in  0.19889533892273903  seconds by  5.684341886080802e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.701392322216066 -56.70139292647929
-------  100 0.4500000000000001 0.7750000000000005
no convergence
weight =  382.42858022284287
set cost params:  1.0 382.42858022284287 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10532.169023638256
Gradient descend method:  None
RUN  1 , total integrated cost =  10532.169023638251


ERROR:root:Problem in initial value trasfer


RUN  2 , total integrated cost =  10532.169023638251
Control only changes marginally.
RUN  2 , total integrated cost =  10532.169023638251
Improved over  2  iterations in  0.13382726907730103  seconds by  5.684341886080802e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.654693677445614 -56.65470553035104
-------  105 0.5750000000000002 0.7750000000000005
converged for  105
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
no convergence
weight =  9342.556622474913
set cost params:  1.0 93

ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  33286.48857826774
Control only changes marginally.
RUN  3 , total integrated cost =  33286.48857826774
Improved over  3  iterations in  0.1863377746194601  seconds by  1.1368683772161603e-13  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354421627345 -56.70354411968511


In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, False], [True, True], [False, False], [False, False], [False, False], [True, True], [True, True], [True, True], [False, False], [False, False], [True, True], [True, True], [True, True], [True, True], [True, False], [True, True], [True, False], [True, False], [False, False]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  158.4917659970987
Gradient descend method:  None
RUN  1 , total integrated cost =  84.27196813305977
RUN  2 , total integrated cost =  79.84447244594925
RUN  3 , total integrated cost =  73.51897602086908
RUN  4 , total integrated cost =  70.56226806826015
RUN  5 , total integrated cost =  64.22131092040912
RUN  6 , total integrated cost =  60.028833482432326
RUN  7 , total integrated cost =  50.88564643007712
RUN  8 , total integrated cost =  45.73865148103937
RUN  9 , total integrated cost =  40.105181724695
RUN  10 , total integrated cost =  38.21767241709237
RUN  11 , total integrated cost =  35.72731570617462
RUN  12 , total integrated cost =  35.12952555547234
RUN  13 , total integrated cost =  33.89727147013476
RUN  14 , total integrated cost =  33.820593001397434
RUN  15 , total integrated cost =  33.490908779561465
RUN  16 , t

ERROR:root:Problem in initial value trasfer


RUN  1000 , total integrated cost =  32.77212178605707
RUN  1000 , total integrated cost =  32.77212178605707
Improved over  1000  iterations in  58.46816426888108  seconds by  79.32250828307573  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446656405857 -56.624466535210715
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  496.12148955303445
Gradient descend method:  HS
RUN  1 , total integrated cost =  494.81519766111694
RUN  2 , total integrated cost =  491.63861288718687
RUN  3 , total integrated cost =  490.0171291443648
RUN  4 , total integrated cost =  489.195855970117
RUN  5 , total integrated cost =  487.99620410597134
RUN  6 , total integrated cost =  487.99620410597083
RUN  7 , total integrated cost =  486.64130455879155
RUN  8 , total integrated cost =  483.27966703718863
RUN  9 , total integrated cost =  479.9946337803611
RUN  10 , total integrated cost =  479.19237859291536
RUN  11 , total in

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  483 , total integrated cost =  278.86941580832115
Improved over  483  iterations in  36.01547931693494  seconds by  43.79009543417277  percent.
Problem in initial value trasfer:  Vmean_exc -56.62442264449522 -56.62442519308031
weight =  1826.8411110178206
set cost params:  1.0 1826.8411110178206 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5090.359384646054
Gradient descend method:  None
RUN  1 , total integrated cost =  5043.328467928173
RUN  2 , total integrated cost =  5022.892536479045
RUN  3 , total integrated cost =  5000.095070954917
RUN  4 , total integrated cost =  4979.732141839038
RUN  5 , total integrated cost =  4954.081127873228
RUN  6 , total integrated cost =  4933.438958648076
RUN  7 , total integrated cost =  4906.347110097283
RUN  8 , total integrated cost =  4884.889087746076
RUN  9 , total integrated cost =  4851.498956021862
RUN  10 , total integrated cost =  4832.060263426955
RUN  11 , total int

...

RUN  10 , total integrated cost =  23280.602006523102
RUN  11 , total integrated cost =  23280.602006523102
Control only changes marginally.
RUN  11 , total integrated cost =  23280.602006523102
Improved over  11  iterations in  1.8224191591143608  seconds by  0.849262402649515  percent.
Problem in initial value trasfer:  Vmean_exc -56.70073509132595 -56.700730815613056
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  393.9113666299245
Gradient descend method:  None
RUN  1 , total integrated cost =  27.099664352968762
RUN  2 , total integrated cost =  27.094124245079858
RUN  3 , total integrated cost =  27.08704645207229
RUN  4 , total integrated cost =  27.08184366249794
RUN  5 , total integrated cost =  27.08167161734493
RUN  6 , total integrated cost =  27.08034980452961
RUN  7 , total integrated cost =  27.07975434666787
RUN  8 , total integrated cost =  27.079614137896602
RUN  

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  26 , total integrated cost =  357.014683593629
Improved over  26  iterations in  2.3276875521987677  seconds by  2.4331525685220186  percent.
Problem in initial value trasfer:  Vmean_exc -56.70353803694884 -56.70353828313358
weight =  9323.56086449655
set cost params:  1.0 9323.56086449655 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33205.03480903065
Gradient descend method:  None
RUN  1 , total integrated cost =  32602.964631554536
RUN  2 , total integrated cost =  32602.938501934907
RUN  3 , total integrated cost =  32602.936870334575
RUN  4 , total integrated cost =  32602.936602534184
RUN  5 , total integrated cost =  32602.936595557858
RUN  6 , total integrated cost =  32602.93659454766
RUN  7 , total integrated cost =  32602.936594411916
RUN  8 , total integrated cost =  32602.93659439484
RUN  9 , total integrated cost =  32602.936594392224
RUN  10 , total integrated cost =  32602.936594391776
RUN  11 , total i

ERROR:root:Problem in initial value trasfer


RUN  13 , total integrated cost =  32602.936594391686
Control only changes marginally.
RUN  13 , total integrated cost =  32602.936594391686
Improved over  13  iterations in  1.1587244160473347  seconds by  1.8132738547083704  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354000048514 -56.70354015869572


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  2793.6654369294015
set cost params:  1.0 2793.6654369294015 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5083.8079477594165
Gradient descend method:  None
RUN  1 , total integrated cost =  5080.659056350758
RUN  2 , total integrated cost =  5080.6549784705185
RUN  3 , total integrated cost =  5080.654974938333
RUN  4 , total integrated cost =  5080.654974933919
RUN  5 , total integrated cost =  5080.654974933915


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  5080.654974933913
RUN  7 , total integrated cost =  5080.654974933913
Control only changes marginally.
RUN  7 , total integrated cost =  5080.654974933913
Improved over  7  iterations in  1.333568001165986  seconds by  0.062019904329645215  percent.
Problem in initial value trasfer:  Vmean_exc -56.62467323738817 -56.624668854224204
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5882.689260195375
set cost params:  1.0 5882.689260195375 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12993.637779553645
Gradient descend method:  None
RUN  1 , total integrated cost =  12991.524282991992
RUN  2 , total integrated cost =  12991.52184466718
RUN  3 , total integrated cost =  12991.521844655796
RUN  4 , total integrated cost =  12991.521844655588
RUN  5 , total integrated cost =  12991.521844655574
RUN  6 , total integrated cost =  12991.52184465557


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  12991.521844655568
RUN  8 , total integrated cost =  12991.521844655568
Control only changes marginally.
RUN  8 , total integrated cost =  12991.521844655568
Improved over  8  iterations in  1.5175686068832874  seconds by  0.016284391899901607  percent.
Problem in initial value trasfer:  Vmean_exc -56.6704058574833 -56.67041169857411
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  1041.4776106837467
set cost params:  1.0 1041.4776106837467 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8216.952113605383
Gradient descend method:  None
RUN  1 , total integrated cost =  8215.544321086782
RUN  2 , total integrated cost =  8215.544321086776
RUN  3 , total integrated cost =  8215.544321086774


...

RUN  1 , total integrated cost =  23521.51645668439
RUN  2 , total integrated cost =  23521.51645668439
Control only changes marginally.
RUN  2 , total integrated cost =  23521.51645668439
Improved over  2  iterations in  0.37837585620582104  seconds by  4.263256414560601e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.70073489405066 -56.70073062610483
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 8
[[True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, False], [True, True], [True, False], [True, True], [True, True], [True, True], [True, False], [True, True], [True, False], [True, True], [False, False], [True, True], [True, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.45000000

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  20061.071797331595
Control only changes marginally.
RUN  1 , total integrated cost =  20061.071797331595
Improved over  1  iterations in  0.2066267430782318  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.695206332122794 -56.69520415021373
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
no convergence
weight =  2115.303938463211
set cost params:  1.0 2115.303938463211 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23521.51645668439
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  23521.51645668439
Control only changes marginally.
RUN  1 , total integrated cost =  23521.51645668439
Improved over  1  iterations in  0.2680841274559498  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70073489405066 -56.70073062610483
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 9
[[True, True], [True, True], [True, True], [True, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
c

In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  38.26858056987436
Gradient descend method:  None
RUN  1 , total integrated cost =  5.796254877127113
RUN  2 , total integrated cost =  5.331634058243294
RUN  3 , total integrated cost =  4.785222418964609
RUN  4 , total integrated cost =  4.531909567247265
RUN  5 , total integrated cost =  4.116436093746105
RUN  6 , total integrated cost =  3.8410693188651726
RUN  7 , total integrated cost =  3.234846323029477
RUN  8 , total integrated cost =  2.879056013985594
RUN  9 , total integrated cost =  2.2508231082115047
RUN  10 , total integrated cost =  2.146343759358018
RUN  11 , total integrated cost =  1.9461888351739822
RUN  12 , total integrated cost =  1.9140484079051303
RUN  13 , total integrated cost =  1.9091739576097524
RUN  14 , total integrated cost =  1.9049213267400862
RUN  15 , total integrated cost =  1.8973798952840146
RUN  1

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  239 , total integrated cost =  1.8497257459829748
Improved over  239  iterations in  6.748364150524139  seconds by  95.16646366696153  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446667207698 -56.62446664413358
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  137.11768238740302
Gradient descend method:  None
RUN  1 , total integrated cost =  2.9825987189440504
RUN  2 , total integrated cost =  2.6217766069638806
RUN  3 , total integrated cost =  2.488533291560394
RUN  4 , total integrated cost =  2.4346258637170193
RUN  5 , total integrated cost =  2.3991942674812554
RUN  6 , total integrated cost =  2.380396496671022
RUN  7 , total integrated cost =  2.3644947595984545
RUN  8 , total integrated cost =  2.3547244993019047
RUN  9 , total integrated cost =  2.3452498048839345
RUN  10 , total integrated cost =  2.338838842240494
RUN  

ERROR:root:Problem in initial value trasfer


RUN  120 , total integrated cost =  2.2709381835165177
Control only changes marginally.
RUN  122 , total integrated cost =  2.270938183516516
Improved over  122  iterations in  2.747745990753174  seconds by  98.34380355328618  percent.
Problem in initial value trasfer:  Vmean_exc -56.67067591027547 -56.67067690665965
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  31.621156516270347
Gradient descend method:  None
RUN  1 , total integrated cost =  12.789013509029477
RUN  2 , total integrated cost =  11.811532385300405
RUN  3 , total integrated cost =  11.040186259625514
RUN  4 , total integrated cost =  10.843154660882005
RUN  5 , total integrated cost =  10.584416406829392
RUN  6 , total integrated cost =  10.453333132498246
RUN  7 , total integrated cost =  10.219897441457517
RUN  8 , total integrated cost =  10.06487470063728
RUN  9 , total integrated cost =  9.73830707167803
RUN  

...

RUN  30 , total integrated cost =  37.28232214977602
RUN  40 , total integrated cost =  36.838369900409596
RUN  50 , total integrated cost =  36.73934040764611
RUN  60 , total integrated cost =  36.62884207523453
RUN  70 , total integrated cost =  36.6038357622829
RUN  80 , total integrated cost =  36.52372009149157
RUN  90 , total integrated cost =  36.406868068765654
RUN  100 , total integrated cost =  36.35028972135929
RUN  110 , total integrated cost =  36.34687770618024
RUN  120 , total integrated cost =  36.33111122758098
RUN  130 , total integrated cost =  36.33019366060637
RUN  140 , total integrated cost =  36.32977853325234
RUN  150 , total integrated cost =  36.32801211277342
RUN  160 , total integrated cost =  36.327786335387145
RUN  170 , total integrated cost =  36.3271824853991
RUN  180 , total integrated cost =  36.32700514283157
RUN  190 , total integrated cost =  36.319477092513175
RUN  200 , total integrated cost =  36.31854411652622


ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  247 , total integrated cost =  36.315225092218974
Improved over  247  iterations in  5.475004270672798  seconds by  13.333052156219836  percent.
Problem in initial value trasfer:  Vmean_exc -56.62417379154703 -56.62417437973956
-------  125 0.47500000000000014 0.8500000000000005
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  55.77927008336469
Gradient descend method:  None
RUN  1 , total integrated cost =  22.788317259692846
RUN  2 , total integrated cost =  22.7614569237474
RUN  3 , total integrated cost =  22.732281066898164
RUN  4 , total integrated cost =  22.722154312876413
RUN  5 , total integrated cost =  22.70718383324682
RUN  6 , total integrated cost =  22.698912953723738
RUN  7 , total integrated cost =  22.677466069325536
RUN  8 , total integrated cost =  22.662949988358626
RUN  9 , total integrated cost =  22.456705927453395
RUN  10 , total integrated cost =  22.354562607733612
RUN

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  139 , total integrated cost =  22.03761149861398
Improved over  139  iterations in  3.081882741302252  seconds by  60.49139498297889  percent.
Problem in initial value trasfer:  Vmean_exc -56.67729552825356 -56.67729551337211
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  152.774852719314
Gradient descend method:  None
RUN  1 , total integrated cost =  12.070407282992113
RUN  2 , total integrated cost =  11.864481520582062
RUN  3 , total integrated cost =  11.735909869731099
RUN  4 , total integrated cost =  11.712436881268108
RUN  5 , total integrated cost =  11.687385939286678
RUN  6 , total integrated cost =  11.678795704331996
RUN  7 , total integrated cost =  11.668228931113727
RUN  8 , total integrated cost =  11.662966228408598
RUN  9 , total integrated cost =  11.654772113611807
RUN  10 , total integrated cost =  11.649154871558828
RUN 

In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

ERROR:root:Problem in initial value trasfer


--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.8497257459829748
Gradient descend method:  None
RUN  1 , total integrated cost =  1.8497257459829748
Control only changes marginally.
RUN  1 , total integrated cost =  1.8497257459829748
Improved over  1  iterations in  0.1100979633629322  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446667207698 -56.62446664413358
-------  15 0.45000

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  2.270938183516516
Control only changes marginally.
RUN  1 , total integrated cost =  2.270938183516516
Improved over  1  iterations in  0.11084757931530476  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67067591027547 -56.67067690665965
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7.936565441262015
Gradient descend method:  None
RUN  1 , total integrated cost =  7.936565441262015
Control only changes marginally.
RUN  1 , total integrated cost =  7.936565441262015
Improved over  1  iterations in  0.08780886046588421  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.639803295936666 -56.63980331425286
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  6.15571969486473

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  10.14412142321886
Control only changes marginally.
RUN  1 , total integrated cost =  10.14412142321886
Improved over  1  iterations in  0.06893221475183964  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69518381047143 -56.695183795565086
-------  75 0.5750000000000002 0.6750000000000004
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.1708899326791402
Gradient descend method:  None
RUN  1 , total integrated cost =  1.1708899326791402
Control only changes marginally.
RUN  1 , total integrated cost =  1.1708899326791402
Improved over  1  iterations in  0.06660925410687923  seconds by  0.0  percent.
-------  85 0.47500000000000014 0.7250000000000004
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  18.414527302852264
Gradient descend method:  None
RUN  1 , total integrated cost =  18.41452730

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  8.907047693793652
Control only changes marginally.
RUN  1 , total integrated cost =  8.907047693793652
Improved over  1  iterations in  0.06796229630708694  seconds by  0.0  percent.
-------  115 0.4250000000000001 0.8250000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  36.315225092218974
Gradient descend method:  None
RUN  1 , total integrated cost =  36.315225092218974
Control only changes marginally.
RUN  1 , total integrated cost =  36.315225092218974
Improved over  1  iterations in  0.06793122366070747  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.62417379154703 -56.62417437973956
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  22.03761149861398
Gradient descend method:  None
RUN  1 , total integrated cost =  22.03761149

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
